In [ ]:
#| default_exp network

# Network

> VPC/Firewall, Secret Manager, Service Accounts, Private Service Connect, Cloud CDN, Cloud Load Balancing, Cloud Armor, IAP, VPC Service Controls.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import time
from gcpeasy._util import (
    _log, translate_error, update_iam_policy, wait_op, wait_rest_op,
)

try:
    from google.cloud import compute_v1
except ImportError:
    pass
try:
    from google.cloud import secretmanager_v1
except ImportError:
    pass
try:
    import googleapiclient.discovery
except ImportError:
    pass

## Constants

In [ ]:
#| export
#: GCP IAP TCP forwarding source range (used for IAP-tunneled SSH/RDP).
IAP_SSH_RANGE = '35.235.240.0/20'
IAP_TCP_RANGES = [IAP_SSH_RANGE]

#: GCP load balancer / health-checker source ranges.
GFE_RANGES = ['130.211.0.0/22', '35.191.0.0/16']

## VPC / Subnet / Firewall

In [ ]:
#| export
def _networks_client(auth):
    return compute_v1.NetworksClient(credentials=auth.credentials)


def create_vpc(auth, name: str, auto_create_subnetworks: bool = False, **_) -> dict:
    """Create a custom-mode VPC network. Returns immediately if already exists."""
    client = _networks_client(auth)
    try:
        existing = client.get(project=auth.project, network=name)
        return {'name': name, 'self_link': existing.self_link}
    except Exception:
        pass

    network = compute_v1.Network(
        name=name,
        auto_create_subnetworks=auto_create_subnetworks,
        routing_config=compute_v1.NetworkRoutingConfig(routing_mode='REGIONAL'),
    )
    op = client.insert(project=auth.project, network_resource=network)
    wait_op(op, what=f'create_vpc {name}', timeout=120)
    existing = client.get(project=auth.project, network=name)
    return {'name': name, 'self_link': existing.self_link}


def add_subnet(auth, network_name: str, subnet_name: str,
               cidr: str = '10.0.0.0/24', region: str = None,
               private_google_access: bool = True, **_) -> dict:
    """Add a subnet to a VPC. ``private_google_access=True`` enables Private Google Access."""
    region = region or auth.region
    client = compute_v1.SubnetworksClient(credentials=auth.credentials)
    try:
        existing = client.get(project=auth.project, region=region, subnetwork=subnet_name)
        return {'name': subnet_name, 'cidr': existing.ip_cidr_range, 'region': region}
    except Exception:
        pass

    subnet = compute_v1.Subnetwork(
        name=subnet_name,
        ip_cidr_range=cidr,
        region=region,
        network=f'projects/{auth.project}/global/networks/{network_name}',
        private_ip_google_access=private_google_access,
    )
    op = client.insert(project=auth.project, region=region, subnetwork_resource=subnet)
    wait_op(op, what=f'add_subnet {subnet_name}', timeout=120)
    return {'name': subnet_name, 'cidr': cidr, 'region': region}


def create_firewall_rule(auth, name: str, network: str = 'default',
                         direction: str = 'INGRESS',
                         protocol: str = 'tcp',
                         ports: list = None,
                         source_ranges: list = None,
                         target_tags: list = None,
                         iap_ssh: bool = False,
                         allow_public: bool = False,
                         **_) -> dict:
    """Create a firewall rule.  Defaults are now **safe**:

    * Ingress with no ``source_ranges`` will raise ``ValueError`` unless
      ``iap_ssh=True`` (uses :data:`IAP_SSH_RANGE`) or
      ``allow_public=True`` (explicit ``0.0.0.0/0``).
    * For SSH from Google IAP, pass ``iap_ssh=True`` (auto-fills
      ``ports=['22']`` when none given).

    Idempotent — returns immediately if the rule already exists.
    """
    client = compute_v1.FirewallsClient(credentials=auth.credentials)
    try:
        client.get(project=auth.project, firewall=name)
        return {'name': name, 'status': 'exists'}
    except Exception:
        pass

    if direction == 'INGRESS' and not source_ranges:
        if iap_ssh:
            source_ranges = list(IAP_TCP_RANGES)
            ports = ports or ['22']
        elif allow_public:
            source_ranges = ['0.0.0.0/0']
        else:
            raise ValueError(
                'create_firewall_rule: INGRESS rules require source_ranges. '
                'Pass iap_ssh=True for IAP-tunneled SSH (35.235.240.0/20), '
                'or allow_public=True to explicitly allow 0.0.0.0/0.'
            )

    allowed = compute_v1.Allowed(I_p_protocol=protocol, ports=ports or [])
    rule = compute_v1.Firewall(
        name=name,
        network=f'projects/{auth.project}/global/networks/{network}',
        direction=direction,
        allowed=[allowed],
        source_ranges=source_ranges or [],
        target_tags=target_tags or [],
    )
    op = client.insert(project=auth.project, firewall_resource=rule)
    wait_op(op, what=f'create_firewall_rule {name}', timeout=60)
    return {'name': name}


def delete_firewall_rule(auth, name: str) -> dict:
    """Delete a firewall rule. Idempotent (returns ``not_found`` if missing)."""
    client = compute_v1.FirewallsClient(credentials=auth.credentials)
    try:
        op = client.delete(project=auth.project, firewall=name)
    except Exception:
        return {'name': name, 'status': 'not_found'}
    wait_op(op, what=f'delete_firewall_rule {name}', timeout=60)
    return {'name': name, 'status': 'deleted'}


def delete_vpc(auth, name: str) -> dict:
    """Delete a VPC network (must be empty of subnets)."""
    client = _networks_client(auth)
    try:
        op = client.delete(project=auth.project, network=name)
    except Exception:
        return {'name': name, 'status': 'not_found'}
    wait_op(op, what=f'delete_vpc {name}', timeout=120)
    return {'name': name, 'status': 'deleted'}


def delete_subnet(auth, name: str, region: str = None) -> dict:
    """Delete a subnet."""
    region = region or auth.region
    client = compute_v1.SubnetworksClient(credentials=auth.credentials)
    try:
        op = client.delete(project=auth.project, region=region, subnetwork=name)
    except Exception:
        return {'name': name, 'status': 'not_found'}
    wait_op(op, what=f'delete_subnet {name}', timeout=120)
    return {'name': name, 'status': 'deleted'}

## Secret Manager

In [ ]:
#| export
def _sm(auth):
    return secretmanager_v1.SecretManagerServiceClient(credentials=auth.credentials)


def create_secret(auth, name: str, value: str, labels: dict = None, **_) -> dict:
    """Create or update a Secret Manager secret. Adds a new version with ``value``.

    When the secret already exists, labels on the secret resource are
    reconciled (added/overwritten) before a new version is added.
    """
    client = _sm(auth)
    parent = f'projects/{auth.project}'
    secret_id = name.replace('/', '-')
    full = f'{parent}/secrets/{secret_id}'

    try:
        existing = client.get_secret(name=full)
        if labels:
            wanted = {**dict(existing.labels), **labels}
            if wanted != dict(existing.labels):
                from google.protobuf import field_mask_pb2
                existing.labels.clear()
                existing.labels.update(wanted)
                client.update_secret(
                    secret=existing,
                    update_mask=field_mask_pb2.FieldMask(paths=['labels']),
                )
    except Exception:
        client.create_secret(
            parent=parent,
            secret_id=secret_id,
            secret=secretmanager_v1.Secret(
                replication=secretmanager_v1.Replication(
                    automatic=secretmanager_v1.Replication.Automatic()
                ),
                labels=labels or {},
            ),
        )

    version = client.add_secret_version(
        parent=full,
        payload=secretmanager_v1.SecretPayload(data=value.encode()),
    )
    return {'name': full, 'version': version.name}


def get_secret(auth, name: str, version: str = 'latest') -> str:
    """Retrieve the value of a Secret Manager secret version."""
    client = _sm(auth)
    secret_id = name.replace('/', '-')
    full = f'projects/{auth.project}/secrets/{secret_id}/versions/{version}'
    response = client.access_secret_version(name=full)
    return response.payload.data.decode()


def update_secret(auth, name: str, value: str):
    "Add a new version to an existing secret."
    create_secret(auth, name, value)


def secret_name(auth, name: str) -> str:
    "Return the full Secret Manager resource name."
    secret_id = name.replace('/', '-')
    return f'projects/{auth.project}/secrets/{secret_id}'


def delete_secret(auth, name: str) -> dict:
    """Delete a Secret Manager secret (all versions). Idempotent."""
    client = _sm(auth)
    secret_id = name.replace('/', '-')
    full = f'projects/{auth.project}/secrets/{secret_id}'
    try:
        client.delete_secret(name=full)
    except Exception:
        return {'name': full, 'status': 'not_found'}
    return {'name': full, 'status': 'deleted'}

## IAM

In [ ]:
#| export
def _iam(auth):
    return googleapiclient.discovery.build('iam', 'v1', credentials=auth.credentials)


def _crm(auth):
    return googleapiclient.discovery.build('cloudresourcemanager', 'v1',
                                           credentials=auth.credentials)


def create_service_account(auth, name: str, display_name: str = '', **_) -> dict:
    """Create a GCP service account. Returns existing account if already present."""
    iam = _iam(auth)
    project_name = f'projects/{auth.project}'
    email = f'{name}@{auth.project}.iam.gserviceaccount.com'
    try:
        existing = iam.projects().serviceAccounts().get(
            name=f'{project_name}/serviceAccounts/{email}'
        ).execute()
        return {'email': existing['email'], 'name': existing['name']}
    except Exception:
        pass
    body = {'accountId': name, 'serviceAccount': {'displayName': display_name or name}}
    result = iam.projects().serviceAccounts().create(name=project_name, body=body).execute()
    return {'email': result['email'], 'name': result['name']}


def bind_iam_role(auth, member_email: str, role: str,
                  member_type: str = 'serviceAccount') -> dict:
    """Bind a project-level IAM role to a member.  Etag-aware, retries on 409."""
    crm = _crm(auth)
    member = f'{member_type}:{member_email}'

    def _mutate(policy):
        for binding in policy.get('bindings', []):
            if binding.get('role') == role and 'condition' not in binding:
                if member in binding.get('members', []):
                    return False
                binding.setdefault('members', []).append(member)
                return True
        policy.setdefault('bindings', []).append(
            {'role': role, 'members': [member]}
        )
        return True

    update_iam_policy(crm, auth.project, _mutate)
    return {'role': role, 'member': member}


def sa_email(auth, name: str) -> str:
    "Return the full email address for a service account in this project."
    return f'{name}@{auth.project}.iam.gserviceaccount.com'


def delete_service_account(auth, email: str) -> dict:
    """Delete a service account by email. Idempotent."""
    iam = _iam(auth)
    full = f'projects/{auth.project}/serviceAccounts/{email}'
    try:
        iam.projects().serviceAccounts().delete(name=full).execute()
    except Exception:
        return {'email': email, 'status': 'not_found'}
    return {'email': email, 'status': 'deleted'}

## Private Service Connect

In [ ]:
#| export
def create_private_service_connect(auth, name: str, network: str, subnet: str,
                                   service_attachment: str, ip_address: str = None,
                                   **_) -> dict:
    """Create a PSC forwarding rule for a managed GCP service."""
    fr_client = compute_v1.ForwardingRulesClient(credentials=auth.credentials)
    try:
        existing = fr_client.get(project=auth.project, region=auth.region,
                                 forwarding_rule=name)
        return {'name': name, 'ip': existing.I_p_address}
    except Exception:
        pass
    body = compute_v1.ForwardingRule(
        name=name,
        network=f'projects/{auth.project}/global/networks/{network}',
        subnetwork=(f'projects/{auth.project}/regions/{auth.region}'
                    f'/subnetworks/{subnet}'),
        target=service_attachment,
        load_balancing_scheme='',
        I_p_address=ip_address,
    )
    op = fr_client.insert(project=auth.project, region=auth.region,
                          forwarding_rule_resource=body)
    wait_op(op, what=f'create_private_service_connect {name}', timeout=120)
    return {'name': name}

## Load balancer plumbing

In [ ]:
#| export
def create_cdn_backend(auth, name: str, bucket_name: str,
                       cdn_policy: dict = None, **_) -> dict:
    """Create a Cloud CDN backend bucket for serving static content."""
    client = compute_v1.BackendBucketsClient(credentials=auth.credentials)
    try:
        existing = client.get(project=auth.project, backend_bucket=name)
        return {'name': name, 'self_link': existing.self_link}
    except Exception:
        pass
    backend = compute_v1.BackendBucket(
        name=name,
        bucket_name=bucket_name,
        enable_cdn=True,
        cdn_policy=compute_v1.BackendBucketCdnPolicy(**(cdn_policy or {})),
    )
    op = client.insert(project=auth.project, backend_bucket_resource=backend)
    wait_op(op, what=f'create_cdn_backend {name}', timeout=120)
    existing = client.get(project=auth.project, backend_bucket=name)
    return {'name': name, 'self_link': existing.self_link}


def reserve_global_ip(auth, name: str, **_) -> dict:
    """Reserve a global static external IPv4 address.  Idempotent."""
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials)
    project = auth.project
    try:
        existing = compute.globalAddresses().get(
            project=project, address=name).execute()
        return {'name': name, 'address': existing['address'],
                'self_link': existing['selfLink']}
    except Exception:
        pass
    op = compute.globalAddresses().insert(project=project, body={
        'name': name, 'addressType': 'EXTERNAL', 'ipVersion': 'IPV4',
    }).execute()
    wait_rest_op(compute, project, op, what=f'reserve_global_ip {name}', timeout=120)
    existing = compute.globalAddresses().get(project=project, address=name).execute()
    return {'name': name, 'address': existing['address'],
            'self_link': existing['selfLink']}


def create_health_check(auth, name: str, port: int = 80, path: str = '/',
                        protocol: str = 'HTTP', **_) -> dict:
    """Create a global health check (HTTP by default).  Idempotent."""
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials)
    project = auth.project
    try:
        existing = compute.healthChecks().get(
            project=project, healthCheck=name).execute()
        return {'name': name, 'self_link': existing['selfLink']}
    except Exception:
        pass
    proto = protocol.upper()
    body = {'name': name, 'type': proto}
    spec = {'port': port}
    if path:
        spec['requestPath'] = path
    body[f'{proto.lower()}HealthCheck'] = spec
    op = compute.healthChecks().insert(project=project, body=body).execute()
    wait_rest_op(compute, project, op, what=f'create_health_check {name}', timeout=60)
    existing = compute.healthChecks().get(
        project=project, healthCheck=name).execute()
    return {'name': name, 'self_link': existing['selfLink']}


def create_backend_service(auth, name: str, health_check: str = None,
                           backends: list = None, protocol: str = 'HTTPS',
                           armor_policy: str = None, port_name: str = 'https',
                           timeout_sec: int = 30, **_) -> dict:
    """Create a global backend service with optional Cloud Armor.

    ``health_check`` accepts either a full self_link or a bare name (which
    is resolved into a global health check self_link).  ``backends`` is a
    list of dicts as the GCP API expects (e.g. ``[{'group': '...neg...'}]``).
    Cloud Armor is attached **here** (not on the forwarding rule), which is
    where GCP actually applies the policy.
    """
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials)
    project = auth.project
    if health_check and not health_check.startswith(('http', 'projects/', 'global/')):
        hc = create_health_check(auth, health_check)
        health_check = hc['self_link']

    try:
        existing = compute.backendServices().get(
            project=project, backendService=name).execute()
        if armor_policy and existing.get('securityPolicy') != armor_policy:
            compute.backendServices().setSecurityPolicy(
                project=project, backendService=name,
                body={'securityPolicy': armor_policy},
            ).execute()
        return {'name': name, 'self_link': existing['selfLink']}
    except Exception:
        pass

    body = {
        'name': name,
        'protocol': protocol,
        'portName': port_name,
        'timeoutSec': timeout_sec,
        'loadBalancingScheme': 'EXTERNAL_MANAGED',
        'healthChecks': [health_check] if health_check else [],
        'backends': backends or [],
    }
    if armor_policy:
        body['securityPolicy'] = armor_policy
    op = compute.backendServices().insert(project=project, body=body).execute()
    wait_rest_op(compute, project, op, what=f'create_backend_service {name}', timeout=120)
    existing = compute.backendServices().get(
        project=project, backendService=name).execute()
    return {'name': name, 'self_link': existing['selfLink']}


def create_https_lb(auth, name: str, backend_service: str,
                    armor_policy: str = None, ssl_certificates: list = None,
                    static_ip: str = None, http_redirect: bool = True,
                    **_) -> dict:
    """Create a global HTTPS load balancer.

    Orchestrates: optional global IP reservation → URL map → target HTTPS
    proxy → global forwarding rule.  Optionally creates an HTTP→HTTPS
    redirect (port 80) when ``http_redirect=True``.

    Cloud Armor must be applied to the backend service via
    :func:`create_backend_service` — passing ``armor_policy`` here is now a
    no-op kept for backward compatibility (a warning is emitted).
    """
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials)
    project = auth.project

    if armor_policy:
        _log('create_https_lb: armor_policy on the LB is a no-op; '
             'attach it to the backend service via create_backend_service.')

    # Global IP
    ip_self_link = None
    ip_address = None
    if static_ip:
        if static_ip.startswith(('http', 'projects/')):
            ip_self_link = static_ip
        else:
            res = reserve_global_ip(auth, static_ip)
            ip_self_link = res['self_link']
            ip_address = res['address']

    # URL map
    url_map_name = f'{name}-url-map'
    try:
        compute.urlMaps().get(project=project, urlMap=url_map_name).execute()
    except Exception:
        compute.urlMaps().insert(project=project, body={
            'name': url_map_name, 'defaultService': backend_service,
        }).execute()

    # HTTPS proxy
    proxy_name = f'{name}-https-proxy'
    try:
        compute.targetHttpsProxies().get(
            project=project, targetHttpsProxy=proxy_name).execute()
    except Exception:
        compute.targetHttpsProxies().insert(project=project, body={
            'name': proxy_name,
            'urlMap': f'global/urlMaps/{url_map_name}',
            'sslCertificates': ssl_certificates or [],
        }).execute()

    # Forwarding rule (HTTPS)
    fr_name = f'{name}-fr'
    try:
        fr = compute.globalForwardingRules().get(
            project=project, forwardingRule=fr_name).execute()
        out = {'name': fr_name, 'ip': fr.get('IPAddress')}
    except Exception:
        body = {
            'name': fr_name,
            'target': f'global/targetHttpsProxies/{proxy_name}',
            'portRange': '443',
            'IPProtocol': 'TCP',
            'loadBalancingScheme': 'EXTERNAL_MANAGED',
        }
        if ip_self_link:
            body['IPAddress'] = ip_self_link
        op = compute.globalForwardingRules().insert(project=project, body=body).execute()
        wait_rest_op(compute, project, op, what=f'create_https_lb {fr_name}', timeout=120)
        fr = compute.globalForwardingRules().get(
            project=project, forwardingRule=fr_name).execute()
        out = {'name': fr_name, 'ip': fr.get('IPAddress', ip_address)}

    # HTTP→HTTPS redirect
    if http_redirect:
        redirect_url_map = f'{name}-http-redirect-url-map'
        try:
            compute.urlMaps().get(project=project, urlMap=redirect_url_map).execute()
        except Exception:
            compute.urlMaps().insert(project=project, body={
                'name': redirect_url_map,
                'defaultUrlRedirect': {
                    'httpsRedirect': True,
                    'redirectResponseCode': 'MOVED_PERMANENTLY_DEFAULT',
                    'stripQuery': False,
                },
            }).execute()
        http_proxy = f'{name}-http-proxy'
        try:
            compute.targetHttpProxies().get(
                project=project, targetHttpProxy=http_proxy).execute()
        except Exception:
            compute.targetHttpProxies().insert(project=project, body={
                'name': http_proxy,
                'urlMap': f'global/urlMaps/{redirect_url_map}',
            }).execute()
        http_fr = f'{name}-http-fr'
        try:
            compute.globalForwardingRules().get(
                project=project, forwardingRule=http_fr).execute()
        except Exception:
            body = {
                'name': http_fr,
                'target': f'global/targetHttpProxies/{http_proxy}',
                'portRange': '80',
                'IPProtocol': 'TCP',
                'loadBalancingScheme': 'EXTERNAL_MANAGED',
            }
            if ip_self_link:
                body['IPAddress'] = ip_self_link
            compute.globalForwardingRules().insert(
                project=project, body=body).execute()
        out['http_redirect'] = http_fr

    return out

## Cloud Armor / Managed certs

In [ ]:
#| export
def create_armor_policy(auth, name: str, rules: list = None, **_) -> dict:
    """Create a Cloud Armor security policy."""
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials)
    project = auth.project
    try:
        existing = compute.securityPolicies().get(
            project=project, securityPolicy=name).execute()
        return {'name': name, 'self_link': existing['selfLink']}
    except Exception:
        pass
    default_rules = [{
        'priority': 2147483647,
        'action': 'allow',
        'match': {'versionedExpr': 'SRC_IPS_V1', 'config': {'srcIpRanges': ['*']}},
        'description': 'default allow rule',
    }]
    body = {'name': name, 'rules': rules or default_rules}
    op = compute.securityPolicies().insert(project=project, body=body).execute()
    wait_rest_op(compute, project, op, what=f'create_armor_policy {name}', timeout=120)
    policy = compute.securityPolicies().get(
        project=project, securityPolicy=name).execute()
    return {'name': name, 'self_link': policy['selfLink']}


def create_managed_cert(auth, name: str, domains: list, **_) -> dict:
    """Create a Google-managed SSL certificate."""
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials)
    project = auth.project
    try:
        existing = compute.sslCertificates().get(
            project=project, sslCertificate=name).execute()
        return {'name': name, 'self_link': existing['selfLink'],
                'status': existing.get('managed', {}).get('status')}
    except Exception:
        pass
    body = {'name': name, 'managed': {'domains': domains}, 'type': 'MANAGED'}
    op = compute.sslCertificates().insert(project=project, body=body).execute()
    wait_rest_op(compute, project, op, what=f'create_managed_cert {name}', timeout=120)
    cert = compute.sslCertificates().get(
        project=project, sslCertificate=name).execute()
    return {'name': name, 'self_link': cert['selfLink'],
            'status': cert.get('managed', {}).get('status')}


def wait_managed_cert_active(auth, name: str, timeout: int = 1800,
                             poll: int = 30) -> dict:
    """Poll a managed SSL certificate until it reaches ``ACTIVE`` status.

    Managed certs only become ACTIVE after DNS for all attached domains
    points at the load balancer IP.  This function blocks until that
    happens, the cert enters a ``FAILED_*`` state, or ``timeout`` elapses.
    """
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials)
    project = auth.project
    start = time.monotonic()
    while True:
        cert = compute.sslCertificates().get(
            project=project, sslCertificate=name).execute()
        managed = cert.get('managed') or {}
        status = managed.get('status', 'PROVISIONING')
        domain_status = managed.get('domainStatus', {})
        if status == 'ACTIVE':
            _log(f'managed_cert {name}: ACTIVE')
            return {'name': name, 'status': status, 'domains': domain_status}
        if status.startswith('FAILED'):
            raise RuntimeError(
                f'managed_cert {name} entered {status}; domains={domain_status}'
            )
        if time.monotonic() - start > timeout:
            raise TimeoutError(
                f'managed_cert {name} not ACTIVE after {timeout}s '
                f'(status={status}, domains={domain_status})'
            )
        _log(f'managed_cert {name}: {status} domains={domain_status}')
        time.sleep(poll)

## IAP & OIDC

In [ ]:
#| export
def get_or_create_oauth_brand(auth, support_email: str,
                              application_title: str = None) -> dict:
    """Return the project OAuth brand (the IAP "consent screen"), creating it
    if absent.  Required before any IAP OAuth client can exist.

    ``support_email`` must be either the email of the user running this code
    or a Google Group the user is a member of.  ``application_title``
    defaults to the project ID.
    """
    iap = googleapiclient.discovery.build(
        'iap', 'v1', credentials=auth.credentials)
    parent = f'projects/{auth.project}'
    brands = iap.projects().brands().list(parent=parent).execute().get('brands', [])
    if brands:
        b = brands[0]
        return {'name': b['name'], 'support_email': b.get('supportEmail'),
                'status': 'exists'}
    body = {
        'supportEmail': support_email,
        'applicationTitle': application_title or auth.project,
    }
    b = iap.projects().brands().create(parent=parent, body=body).execute()
    return {'name': b['name'], 'support_email': b.get('supportEmail'),
            'status': 'created'}


def enable_iap(auth, backend_service_name: str, iap_client_id: str,
               iap_client_secret: str, **_) -> dict:
    """Enable IAP on a backend service using a precise field-mask update."""
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials)
    project = auth.project
    body = {
        'iap': {
            'enabled': True,
            'oauth2ClientId': iap_client_id,
            'oauth2ClientSecret': iap_client_secret,
        },
    }
    op = compute.backendServices().patch(
        project=project,
        backendService=backend_service_name,
        body=body,
    ).execute()
    wait_rest_op(compute, project, op, what=f'enable_iap {backend_service_name}',
                 timeout=120)
    return {'backend_service': backend_service_name, 'iap_enabled': True}


def get_oidc_token(auth, target_audience: str) -> str:
    """Fetch a short-lived OIDC ID token for service-to-service authentication."""
    from google.auth.transport.requests import Request
    from google.oauth2 import id_token as _id_token
    return _id_token.fetch_id_token(Request(), target_audience)

## VPC SC

In [ ]:
#| export
def create_vpc_sc_perimeter(auth, policy_resource: str, perimeter_name: str,
                            restricted_services: list = None,
                            access_levels: list = None, **_) -> dict:
    """Create a VPC Service Controls service perimeter."""
    try:
        from google.cloud import accesscontextmanager_v1
    except ImportError:  # pragma: no cover
        raise ImportError(
            'Install google-cloud-access-context-manager: '
            'pip install google-cloud-access-context-manager'
        )

    acm = accesscontextmanager_v1.AccessContextManagerClient(credentials=auth.credentials)
    perimeter_resource = f'{policy_resource}/servicePerimeters/{perimeter_name}'
    try:
        existing = acm.get_service_perimeter(name=perimeter_resource)
        return {'name': existing.name, 'title': existing.title}
    except Exception:
        pass
    _default_services = [
        'aiplatform.googleapis.com', 'storage.googleapis.com',
        'bigquery.googleapis.com', 'secretmanager.googleapis.com',
    ]
    perimeter = accesscontextmanager_v1.ServicePerimeter(
        name=perimeter_resource,
        title=perimeter_name,
        perimeter_type=(
            accesscontextmanager_v1.ServicePerimeter.PerimeterType.PERIMETER_TYPE_REGULAR
        ),
        status=accesscontextmanager_v1.ServicePerimeterConfig(
            resources=[f'projects/{auth.project}'],
            restricted_services=restricted_services or _default_services,
            access_levels=access_levels or [],
        ),
    )
    op = acm.create_service_perimeter(parent=policy_resource,
                                      service_perimeter=perimeter)
    result = wait_op(op, what=f'create_vpc_sc_perimeter {perimeter_name}',
                     timeout=120)
    return {'name': result.name, 'title': perimeter_name}

## Cloud DNS (B3)

In [ ]:
#| export
def create_dns_zone(auth, zone_name: str, dns_name: str,
                    description: str = '', **_) -> dict:
    """Create a public Cloud DNS managed zone.  Idempotent.

    ``dns_name`` must be the FQDN with trailing dot, e.g. ``'example.com.'``.
    """
    dns = googleapiclient.discovery.build(
        'dns', 'v1', credentials=auth.credentials)
    project = auth.project
    if not dns_name.endswith('.'):
        dns_name = dns_name + '.'
    try:
        existing = dns.managedZones().get(
            project=project, managedZone=zone_name).execute()
        return {'name': zone_name, 'dns_name': existing['dnsName'],
                'name_servers': existing.get('nameServers', [])}
    except Exception:
        pass
    body = {
        'name': zone_name, 'dnsName': dns_name,
        'description': description or f'gcpeasy zone for {dns_name}',
        'visibility': 'public',
    }
    z = dns.managedZones().create(project=project, body=body).execute()
    return {'name': zone_name, 'dns_name': z['dnsName'],
            'name_servers': z.get('nameServers', [])}


def cloud_dns_record(auth, zone: str, name: str, type: str,
                     rrdatas: list, ttl: int = 300, **_) -> dict:
    """Upsert a Cloud DNS record.  Replaces any existing record of the same
    ``(name, type)`` pair.

    ``name`` must be a FQDN ending in ``'.'``.  ``rrdatas`` is a list of
    record values (e.g. ``['1.2.3.4']`` for an A record).
    """
    dns = googleapiclient.discovery.build(
        'dns', 'v1', credentials=auth.credentials)
    project = auth.project
    if not name.endswith('.'):
        name = name + '.'
    new_rec = {'name': name, 'type': type, 'ttl': ttl, 'rrdatas': list(rrdatas)}
    deletions = []
    page_token = None
    while True:
        kwargs = dict(project=project, managedZone=zone, name=name, type=type)
        if page_token:
            kwargs['pageToken'] = page_token
        page = dns.resourceRecordSets().list(**kwargs).execute()
        deletions.extend(page.get('rrsets', []))
        page_token = page.get('nextPageToken')
        if not page_token:
            break
    body = {'additions': [new_rec], 'deletions': deletions}
    change = dns.changes().create(project=project, managedZone=zone, body=body).execute()
    return {'zone': zone, 'name': name, 'type': type,
            'change_id': change.get('id'), 'status': change.get('status')}

### Tests — module exports

In [ ]:
#| hide
import gcpeasy.network as M
for n in ['create_vpc', 'create_firewall_rule', 'create_secret',
          'create_service_account', 'bind_iam_role', 'create_https_lb',
          'enable_iap']:
    assert n in M.__all__

### Tests — ported from `tests/test_network.py`


In [ ]:
#| hide
import sys as _sys, types as _types
from unittest.mock import MagicMock, patch
_kernel = _sys.modules['__main__']

class _MockAuth:
    project = 'test-project'
    region = 'us-central1'
    credentials = MagicMock(name='credentials')
def _mock_auth(): return _MockAuth()

def _install_fake_compute_v1():
    mod = _types.ModuleType('google.cloud.compute_v1')
    class _W:
        def __init__(self, **kw):
            for k, v in kw.items(): setattr(self, k, v)
    for name in ['Firewall', 'Allowed', 'NetworkInterface', 'Network',
                 'NetworkRoutingConfig', 'Subnetwork', 'ForwardingRule',
                 'BackendBucket', 'BackendBucketCdnPolicy']:
        setattr(mod, name, type(name, (_W,), {}))
    for cli in ['NetworksClient', 'SubnetworksClient', 'FirewallsClient',
                'ForwardingRulesClient', 'BackendBucketsClient']:
        setattr(mod, cli, MagicMock(name=cli))
    return mod


In [ ]:
#| hide
# A5: ingress without explicit source must be rejected (defense-in-depth)
fake_compute = _install_fake_compute_v1()
with patch.dict(_sys.modules, {'google.cloud.compute_v1': fake_compute}), \
     patch.object(_kernel, 'compute_v1', fake_compute, create=True):
    fake_compute.FirewallsClient.return_value.get.side_effect = Exception('nf')
    try:
        create_firewall_rule(_mock_auth(), 'open-22', network='default',
                             protocol='tcp', ports=['22'])
    except ValueError as e:
        assert 'source_ranges' in str(e)
    else:
        raise AssertionError('expected ValueError')


In [ ]:
#| hide
# iap_ssh=True selects the IAP source range and auto-fills port 22
fake_compute = _install_fake_compute_v1()
with patch.dict(_sys.modules, {'google.cloud.compute_v1': fake_compute}), \
     patch.object(_kernel, 'compute_v1', fake_compute, create=True):
    client = fake_compute.FirewallsClient.return_value
    client.get.side_effect = Exception('nf')
    op = MagicMock(); op.done = MagicMock(return_value=True); op.result = MagicMock()
    client.insert.return_value = op
    create_firewall_rule(_mock_auth(), 'iap-ssh', network='default', iap_ssh=True)
    rule = client.insert.call_args.kwargs['firewall_resource']
    assert rule.source_ranges == [IAP_SSH_RANGE]
    assert rule.allowed[0].ports == ['22']


In [ ]:
#| hide
# allow_public=True opens 0.0.0.0/0 (with caller's explicit consent)
fake_compute = _install_fake_compute_v1()
with patch.dict(_sys.modules, {'google.cloud.compute_v1': fake_compute}), \
     patch.object(_kernel, 'compute_v1', fake_compute, create=True):
    client = fake_compute.FirewallsClient.return_value
    client.get.side_effect = Exception('nf')
    op = MagicMock(); op.done = MagicMock(return_value=True); op.result = MagicMock()
    client.insert.return_value = op
    create_firewall_rule(_mock_auth(), 'pub', network='default', ports=['443'],
                         allow_public=True)
    rule = client.insert.call_args.kwargs['firewall_resource']
    assert rule.source_ranges == ['0.0.0.0/0']


In [ ]:
#| hide
# A6: bind_iam_role uses etag-aware read-modify-write at policy v3
initial = {'bindings': [], 'etag': 'etag-1'}
state = {'set_calls': [], 'policy': dict(initial)}
fake_crm = MagicMock()
def get_(resource, body):
    m = MagicMock()
    m.execute = lambda: dict(state['policy'])
    state['get_body'] = body
    return m
def set_(resource, body):
    state['set_calls'].append(body); state['policy'] = body['policy']
    m = MagicMock(); m.execute = lambda: state['policy']; return m
fake_crm.projects().getIamPolicy.side_effect = get_
fake_crm.projects().setIamPolicy.side_effect = set_

with patch.object(_kernel, '_crm', return_value=fake_crm):
    bind_iam_role(_mock_auth(), 'sa@p.iam.gserviceaccount.com', 'roles/viewer')

assert state['get_body']['options']['requestedPolicyVersion'] == 3
assert len(state['set_calls']) == 1
new = state['set_calls'][0]['policy']
assert new['etag'] == 'etag-1'
assert new['version'] == 3
assert any(b['role'] == 'roles/viewer'
           and 'serviceAccount:sa@p.iam.gserviceaccount.com' in b['members']
           for b in new['bindings'])


In [ ]:
#| hide
# bind_iam_role: idempotent — when member already bound, no setIamPolicy call
state = {'set_calls': [], 'policy': {
    'bindings': [{'role': 'roles/viewer',
                  'members': ['serviceAccount:sa@p.iam.gserviceaccount.com']}],
    'etag': 'e', 'version': 3,
}}
fake_crm = MagicMock()
fake_crm.projects().getIamPolicy.return_value.execute = lambda: dict(state['policy'])
def set_(resource, body):
    state['set_calls'].append(body)
    m = MagicMock(); m.execute = lambda: state['policy']; return m
fake_crm.projects().setIamPolicy.side_effect = set_

with patch.object(_kernel, '_crm', return_value=fake_crm):
    bind_iam_role(_mock_auth(), 'sa@p.iam.gserviceaccount.com', 'roles/viewer')
assert state['set_calls'] == []


In [ ]:
#| hide
# A9: get_or_create_oauth_brand returns existing brand without creating
fake_iap = MagicMock()
fake_iap.projects().brands().list().execute.return_value = {
    'brands': [{'name': 'projects/123/brands/abc', 'supportEmail': 'admin@x.com'}]
}
with patch('googleapiclient.discovery.build', return_value=fake_iap):
    out = get_or_create_oauth_brand(_mock_auth(), support_email='admin@x.com')
assert out['status'] == 'exists'
assert out['name'] == 'projects/123/brands/abc'


In [ ]:
#| hide
# get_or_create_oauth_brand creates when no brand exists
fake_iap = MagicMock()
fake_iap.projects().brands().list().execute.return_value = {'brands': []}
fake_iap.projects().brands().create().execute.return_value = {
    'name': 'projects/123/brands/new', 'supportEmail': 'admin@x.com',
}
with patch('googleapiclient.discovery.build', return_value=fake_iap):
    out = get_or_create_oauth_brand(_mock_auth(), support_email='admin@x.com')
assert out['status'] == 'created'
